# Financial Asset Recommendation System

## Part 3 — Cleaning, formatting and temporal splitting

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.preprocessing import clean_data, temporal_user_split
assets=pd.read_csv(RAW_DATA_DIR/'assets.csv')
users=pd.read_csv(RAW_DATA_DIR/'users.csv')
interactions=pd.read_csv(RAW_DATA_DIR/'interactions.csv')

## 1. Validation and cleaning

The reusable cleaning function checks required columns, referential integrity, duplicate identifiers, timestamps and positive implicit-feedback weights.

In [ ]:
assets,users,interactions=clean_data(assets,users,interactions)
print(assets.shape,users.shape,interactions.shape)

## 2. Temporal user-level split

For every user, the most recent interaction is assigned to test, the preceding interaction to validation, and all earlier interactions to training. This prevents future behavior from leaking into model training.

In [ ]:
train,validation,test=temporal_user_split(interactions,validation_items=1,test_items=1)
summary=pd.DataFrame({'rows':[len(train),len(validation),len(test)],'users':[train.user_id.nunique(),validation.user_id.nunique(),test.user_id.nunique()]},index=['train','validation','test'])
summary

In [ ]:
check=pd.DataFrame({'train_max':train.groupby('user_id').timestamp.max(),'validation_time':validation.groupby('user_id').timestamp.min(),'test_time':test.groupby('user_id').timestamp.min()})
print('Users respecting chronology:',((check.train_max<=check.validation_time)&(check.validation_time<=check.test_time)).mean())

## 3. Save processed data

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True,exist_ok=True)
assets.to_csv(PROCESSED_DATA_DIR/'assets.csv',index=False)
users.to_csv(PROCESSED_DATA_DIR/'users.csv',index=False)
train.to_csv(PROCESSED_DATA_DIR/'train.csv',index=False)
validation.to_csv(PROCESSED_DATA_DIR/'validation.csv',index=False)
test.to_csv(PROCESSED_DATA_DIR/'test.csv',index=False)
print('Saved to',PROCESSED_DATA_DIR)

## 4. Statistical validity and limitations

A chronological split is appropriate for sequential recommendation. However, a single held-out item per user yields a high-variance estimate. Later versions should add repeated temporal backtesting, bootstrap confidence intervals and sensitivity analysis for the interaction-generation rules.